In [2]:
import pandas as pd


df = pd.read_parquet('../Imdb_Movie_Dataset.parquet')
df_aux = df.copy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd
import pickle
import gzip
import gc

df_aux['revenue'] = pd.to_numeric(df_aux['revenue'], errors='coerce')
df_aux['runtime'] = pd.to_numeric(df_aux['runtime'], errors='coerce')
df_aux['budget'] = pd.to_numeric(df_aux['budget'], errors='coerce')
df_aux['popularity'] = pd.to_numeric(df_aux['popularity'], errors='coerce')
df_aux['vote_average'] = pd.to_numeric(df_aux['vote_average'], errors='coerce')
df_aux['vote_count'] = pd.to_numeric(df_aux['vote_count'], errors='coerce')

df_clean = df_aux[
    df_aux['vote_average'].notna() & 
    (df_aux['vote_average'] > 0) & 
    df_aux['vote_count'].notna() &
    (df_aux['vote_count'] >= 10)
].copy()

print(f"Filmes com relevância estatística para treinamento (>= 10 votos): {len(df_clean)}")

df_clean['release_year'] = pd.to_datetime(df_clean['release_date'], errors='coerce').dt.year
df_clean['release_decade'] = (df_clean['release_year'] // 10) * 10

df_clean['overview_len'] = df_clean['overview'].astype(str).str.len()
df_clean['tagline_len'] = df_clean['tagline'].astype(str).str.len()

df_clean['keywords'] = df_clean['keywords'].astype(str).fillna('')
df_clean['is_short_keyword'] = df_clean['keywords'].str.contains('short', case=False, regex=False).astype('int8')

df_clean['has_budget'] = (df_clean['budget'] > 0).astype('int8')
df_clean['has_revenue'] = (df_clean['revenue'] > 0).astype('int8')

df_clean['genres'] = df_clean['genres'].astype(str).fillna('')
df_clean['main_genre'] = df_clean['genres'].str.split(', ').str[0]

global_vote_mean = df_clean['vote_average'].mean()
genre_stats = df_clean.groupby('main_genre')['vote_average'].agg(['mean', 'count'])

smoothing_g = 15
df_clean['genre_vote_mean'] = df_clean['main_genre'].map(
    lambda x: (genre_stats.loc[x, 'mean'] * genre_stats.loc[x, 'count'] + global_vote_mean * smoothing_g) / (genre_stats.loc[x, 'count'] + smoothing_g) if x in genre_stats.index else global_vote_mean
)

genre_decade_stats = df_clean.groupby(['main_genre', 'release_decade'])['vote_average'].agg(['mean', 'count'])
smoothing_gd = 20
def get_smooth_genre_decade_vote(row):
    key = (row['main_genre'], row['release_decade'])
    if key in genre_decade_stats.index:
        stat = genre_decade_stats.loc[key]
        return (stat['mean'] * stat['count'] + row['genre_vote_mean'] * smoothing_gd) / (stat['count'] + smoothing_gd)
    return row['genre_vote_mean']

df_clean['genre_decade_vote_mean'] = df_clean.apply(get_smooth_genre_decade_vote, axis=1)

df_clean['log_vote_count'] = np.log1p(df_clean['vote_count'])
df_clean['budget_revenue_ratio'] = df_clean['budget'] / (df_clean['revenue'] + 1)
df_clean['budget_to_popularity'] = df_clean['budget'] / (df_clean['popularity'] + 1)
df_clean['popularity_to_votes'] = df_clean['popularity'] / (df_clean['vote_count'] + 1)
df_clean['votes_per_year'] = df_clean['vote_count'] / (2027 - df_clean['release_year'] + 1)
df_clean['popularity_log_votes_interact'] = df_clean['popularity'] * df_clean['log_vote_count']
df_clean['is_en'] = (df_clean['original_language'] == 'en').astype('int8')

numerical_features = [
    'runtime', 'vote_count', 'log_vote_count', 'release_year', 'release_decade', 'budget', 
    'popularity', 'is_short_keyword', 'overview_len', 'tagline_len', 
    'genre_vote_mean', 'genre_decade_vote_mean', 'budget_revenue_ratio', 
    'budget_to_popularity', 'popularity_to_votes', 'has_budget', 
    'has_revenue', 'is_en', 'votes_per_year', 'popularity_log_votes_interact'
]
multi_value_categorical_features = ['genres', 'production_companies', 'production_countries']
single_value_categorical_features = []

all_base_features = numerical_features + multi_value_categorical_features + single_value_categorical_features + ['main_genre']

df_train_local = df_clean[all_base_features + ['vote_average']].copy()
df_train_local.dropna(subset=['vote_count', 'release_year', 'budget', 'popularity', 'runtime'], inplace=True)

top_n_categories = 50
cols_to_drop = ['main_genre']
new_features_dict = {}

for col in multi_value_categorical_features:
    df_train_local[col] = df_train_local[col].astype(str).fillna('')
    item_counts = df_train_local[col].str.split(', ').explode().str.strip().value_counts()
    top_items = item_counts[item_counts.index != ''].head(top_n_categories).index.tolist()
    
    for item_name in top_items:
        new_features_dict[f'{col}_{item_name}'] = df_train_local[col].str.contains(item_name, regex=False, na=False).astype('int8')
    
    cols_to_drop.append(col)

df_new_features = pd.DataFrame(new_features_dict, index=df_train_local.index)
df_train_local = pd.concat([df_train_local, df_new_features], axis=1)
df_train_local.drop(columns=cols_to_drop, inplace=True, errors='ignore')

df_train_local = pd.get_dummies(df_train_local, columns=[col for col in single_value_categorical_features], drop_first=True)

print(f"Número total de features criadas: {len(df_train_local.columns) - 1}")

X = df_train_local.drop('vote_average', axis=1)
y = df_train_local['vote_average']

features_for_prediction_final = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

del df_clean, df_train_local, X, y, df_new_features, new_features_dict
gc.collect()

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=5,
    max_features=0.4,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"\n--- Avaliação do Modelo Random Forest para Previsão de Notas (Vote Average) ---")
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:,.2f} pontos na nota")
print(f"MAE: {mae:,.2f} pontos na nota")
print(f"MAPE: {mape * 100:.2f}%")

print("\n--- Top 20 Importância das Features ---")
feature_importances = sorted(zip(features_for_prediction_final, model.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in feature_importances[:20]:
    print(f"{feat}: {imp:.4f}")

with gzip.open('models/randomforest_vote_average_model.pkl.gz', 'wb') as f:
    pickle.dump(model, f)
    
with open('models/features_rf_vote_average_model.pkl', 'wb') as f:
    pickle.dump(features_for_prediction_final, f)

Filmes com relevância estatística para treinamento (>= 10 votos): 78678
Número total de features criadas: 140

--- Avaliação do Modelo Random Forest para Previsão de Notas (Vote Average) ---
R² Score: 0.4549
RMSE: 0.76 pontos na nota
MAE: 0.57 pontos na nota
MAPE: 10.24%

--- Top 20 Importância das Features ---
genre_decade_vote_mean: 0.1773
runtime: 0.1153
vote_count: 0.0629
log_vote_count: 0.0617
genres_Horror: 0.0610
popularity_to_votes: 0.0596
release_year: 0.0481
votes_per_year: 0.0474
genre_vote_mean: 0.0453
popularity_log_votes_interact: 0.0396
overview_len: 0.0341
popularity: 0.0338
is_en: 0.0293
tagline_len: 0.0197
release_decade: 0.0152
genres_Drama: 0.0120
production_countries_United States of America: 0.0120
budget_to_popularity: 0.0111
genres_Science Fiction: 0.0084
genres_Documentary: 0.0079
